*Cellule 1 — Vérifier l'environnement*

In [1]:
import openai, pandas as pd

print("openai version:", openai.__version__)
print("pandas version:", pd.__version__)

openai version: 3.26.0
pandas version: 2.2.3


*Cellule 2 — Connexion au serveur*

In [2]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

models = client.models.list()
print("Modèles disponibles :")
for m in models.data:
    print(" -", m.id)

Modèles disponibles :
 - mistralai/mistral-7b-instruct-v0.3
 - google/gemma-4-e4b
 - text-embedding-nomic-embed-text-v1.5


*Cellule 3 — Un tout premier échange*

In [3]:
response = client.chat.completions.create(
    model="mistralai/mistral-7b-instruct-v0.3",
    messages=[{"role": "user", "content": "Say hello in one word."}],
)
print("Réponse :", response.choices[0].message.content)

Réponse :  Hello!


*Cellule 4 — Un prompt un peu plus long (mi-chemin)*

In [4]:
response = client.chat.completions.create(
    model="mistralai/mistral-7b-instruct-v0.3",
    messages=[
        {"role": "system", "content": "You are a supply chain analyst. Be concise and factual."},
        {"role": "user", "content": "A supplier has 15 delivery delays this month and 3 quality issues. Summarize in one sentence."},
    ],
    temperature=0.2,
    max_tokens=100,
)
print("Réponse :", response.choices[0].message.content)

BadRequestError: Error code: 400 - {'error': 'Error rendering prompt with jinja template: "Only user and assistant roles are supported!".\n\nThis is usually an issue with the model\'s prompt template. If you are using a popular model, you can try to search the model under lmstudio-community, which will have fixed prompt templates. If you cannot find one, you are welcome to post this issue to our discord or issue tracker on GitHub. Alternatively, if you know how to write jinja templates, you can override the prompt template in My Models > model settings > Prompt Template.'}

*Cellule 5 — Charger tes données*

In [5]:
import sys, os
sys.path.append(os.path.abspath("../scripts"))

df = pd.read_csv("../data/cleaned_comments.csv")
print("Lignes :", len(df))

Lignes : 1610


*Cellule 6 — Calculer les stats*

In [6]:
from stats_engine import compute_stats

stats = compute_stats(df)
print("total_comments :", stats["total_comments"])
print("Clés :", list(stats.keys()))

total_comments : 1610
Clés : ['total_comments', 'top_root_causes', 'top_root_causes_pct', 'first_snapshot', 'last_snapshot', 'comments_first_snapshot', 'comments_last_snapshot', 'trend_pct', 'causes_increasing', 'causes_decreasing', 'material_status_counts', 'impact_counts', 'top_comments', 'vendor_count']


*Cellule 7 — Construire le prompt (sans appeler le serveur)*

In [7]:
from llm_summary import build_prompt

sample_comments = df["Comment_EN"].value_counts().head(5).index.tolist()
messages = build_prompt(stats, sample_comments)

print("Nb de messages :", len(messages))
print("Taille totale (caractères) :", sum(len(m["content"]) for m in messages))
print("\n--- PROMPT SYSTÈME ---\n", messages[0]["content"])
print("\n--- PROMPT UTILISATEUR (800 premiers caractères) ---\n", messages[1]["content"][:800])

Nb de messages : 1
Taille totale (caractères) : 2121

--- PROMPT SYSTÈME ---
 You are a supply chain analyst at a pharmaceutical company. You receive PRECOMPUTED statistics and sample planner comments. Write a concise analysis in 3 to 5 sentences covering: (1) the main issues, (2) how the situation evolved over time, (3) any emerging risk worth escalating. STRICT RULES: only use the numbers provided in the statistics — never compute, estimate, or invent figures. Do not use markdown formatting. Write in English.

Statistics (computed by pandas, JSON):
{
  "total_comments": 1610,
  "top_root_causes": {
    "PL_Shortterm demand": 283,
    "S_Lack of Raw Material": 166,
    "PL_Internal Errors": 121,
    "S_Quality Failed Finished Goods": 72,
    "S_Machine/Capacity Issues": 63
  },
  "top_root_causes_pct": {
    "PL_Shortterm demand": 17.6,
    "S_Lack of Raw Material": 10.3,
    "PL_Internal Errors": 7.5,
    "S_Quality Failed Finished Goods": 4.5,
    "S_Machine/Capacity Issues": 3.9
  

IndexError: list index out of range

*Cellule 8 — L'appel final (le vrai test)*

In [8]:
from llm_summary import generate_summary

summary = generate_summary(stats, sample_comments)
print("\n=== AI SUMMARY ===")
print(summary)

TypeError: 'NoneType' object is not subscriptable